In [1]:
!pip install -q transformers accelerate sentence-transformers faiss-cpu torch


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import re
from pathlib import Path

docs = []

for file in Path("dataset").rglob("*.txt"):

    text = file.read_text(encoding="utf-8")

    # split into TEXT blocks
    chunks = re.split(r"(?=TEXT\s+\d+)", text)

    for chunk in chunks:

        chunk = chunk.strip()

        if not chunk.startswith("TEXT"):
            continue

        m = re.match(r"TEXT\s+(\d+)", chunk)

        if not m:
            continue

        verse_no = int(m.group(1))

        docs.append({
            "path": str(file),
            "verse": verse_no,
            "text": chunk
        })

print("documents =", len(docs))

documents = 627


In [3]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(
    "BAAI/bge-base-en-v1.5"
)

c:\Users\rakes\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2721.90it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
texts = [d["text"] for d in docs]

embeddings = embedder.encode(
    texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

Batches: 100%|██████████| 20/20 [01:53<00:00,  5.68s/it]


In [5]:
import faiss
import pickle

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

faiss.write_index(index, "verses.faiss")

with open("docs.pkl", "wb") as f:
    pickle.dump(docs, f)

print("saved")

saved


In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen3-1.7B-Base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1816.30it/s]
Some parameters are on the meta device because they were offloaded to the cpu and disk.


In [7]:
import faiss
import pickle

index = faiss.read_index("verses.faiss")

with open("docs.pkl", "rb") as f:
    docs = pickle.load(f)

def retrieve(query, k=5):

    q = embedder.encode(
        [query],
        convert_to_numpy=True
    )

    D, I = index.search(q, k)

    return [docs[i] for i in I[0]]

In [8]:
def ask(question):
    print("Question:", question)
    results = retrieve(question, k=3)  # reduced from 5 → 3

    context = "\n\n".join(
        r["text"] for r in results
    )
    print("Results retrieved:", results)
    prompt = f"""
Answer ONLY from the context below.
Do not use outside knowledge.

Context:
{context} - {results[0]['path']}

Question:
{question}

Answer:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048   # 🔥 CRITICAL FIX
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=250
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [ ]:
#while True:

# q = "WHAT LORD KRISHNA INSTRUCTS ON SURRENDERING?" #input("Question: ")
q = input("Question: ")


#if q.lower() == "exit":
 #   break

print("\n", ask(q), "\n")

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Question: WHAT LORD KRISHNA INSTRUCTS ON SURRENDERING?
Results retrieved: [{'path': 'dataset\\bg\\canto1\\chapter18.txt', 'verse': 66, 'text': "TEXT 66\n\nTranslation\nAbandon all varieties of religion and just surrender unto Me. I shall deliver you from all sinful reactions. Do not fear.\n\nPurport\nThe Lord has described various kinds of knowledge and processes of religion – knowledge of the Supreme Brahman, knowledge of the Supersoul, knowledge of the different types of orders and statuses of social life, knowledge of the renounced order of life, knowledge of nonattachment, sense and mind control, meditation, etc. He has described in so many ways different types of religion. Now, in summarizing Bhagavad-gita, the Lord says that Arjuna should give up all the processes that have been explained to him; he should simply surrender to Krishna. That surrender will save him from all kinds of sinful reactions, for the Lord personally promises to protect him.\n\nIn the Seventh Chapter it was 

KeyboardInterrupt: 